# 🔥 PyTorch Workflow — Linear Regression 



### What you'll find here
- 📝 A markdown explanation **before every code cell**, describing what it does and why
- 📚 Reference tables for every built-in PyTorch/Python function used, right where they first appear
- 📋 A full **cheat-sheet** of every function/class used, at the very end, for quick revision

### The general PyTorch workflow

```
1. Get data ready (turn it into tensors)
2. Build or pick a pretrained model
        - Pick a loss function & optimizer
        - Build a training loop
3. Fit the model to the data (training)
4. Make predictions and evaluate the model (inference)
5. Save and load the model
6. Put it all together
```

This notebook implements this workflow **twice**:
- **Part 1** — builds a linear regression model "from scratch" using raw `nn.Parameter` tensors, so you can see exactly what's happening under the hood.
- **Part 2 ("Putting it all together")** — rebuilds the same idea the *idiomatic* way, using `nn.Linear` and device-agnostic code, which is how you'd actually write PyTorch code in practice.

> 💡 **Tip:** Run cells in order, top to bottom (`Runtime → Run all` in Colab). Several cells depend on variables created earlier (e.g. `model_0`, `X_train`).


## 0. Setup & Imports

We only need three things for this workflow: `torch` itself, the `nn` submodule (neural-network building blocks like `nn.Module`, `nn.Linear`, `nn.Parameter`), and `matplotlib` for plotting.

| Import | Purpose |
|---|---|
| `torch` | Core tensor library — creating tensors, math ops, autograd |
| `torch.nn` | Building blocks for models: `nn.Module`, `nn.Parameter`, `nn.Linear`, loss functions |
| `matplotlib.pyplot` | Plotting training data and predictions |

> The original notebook also imported `DataLoader`, `torchvision.datasets`, and some transforms — those belong to an image-classification workflow and are unused here, so they've been removed.


In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")

---
# Part 1 — Linear Regression "from scratch"

We're going to teach a model to learn the relationship `y = weight * X + bias` **without telling it `weight` and `bias` directly**. It will start with random guesses and use gradient descent to converge on the right values.


## 1.1 Create known parameters & synthetic data

We pick a `weight` and `bias` ourselves (the "ground truth"), then generate `X` values and compute the matching `y = weight * X + bias`. The model's whole job will be to *rediscover* `0.7` and `0.3` purely from `(X, y)` pairs.

| Function | Signature | What it does |
|---|---|---|
| `torch.arange(start, end, step)` | returns a 1‑D tensor of values `[start, end)` spaced by `step` | Here: `torch.arange(0, 1, 0.02)` → 50 evenly spaced values from 0.0 to 0.98 |
| `Tensor.unsqueeze(dim)` | inserts a new size-1 dimension at position `dim` | Turns shape `[50]` into `[50, 1]` — models expect a "batch of feature vectors" (2‑D), not a flat 1‑D list |


In [ ]:
# Create *known* parameters (the "ground truth" the model must learn)
weight = 0.7
bias = 0.3

# Create data
start = 0
end = 1
step = 0.02
X = torch.arange(start, end, step).unsqueeze(dim=1)  # shape: [50, 1]
# y = weight * X + bias  (a straight line)
y = weight * X + bias

X[:10], y[:10]

## 1.2 Check the shapes

Always sanity-check tensor shapes before feeding them into a model — shape mismatches are the #1 source of silent bugs in PyTorch.


In [ ]:
print(X.shape)
print(y.shape)

## 1.3 Split into train/test sets

Standard ML practice: train on most of the data, hold out a slice to check the model generalizes. Here we use a simple **80% train / 20% test** split, done with plain tensor slicing (no shuffling needed since `X` is already evenly spaced).


In [ ]:
# Create train/test split (80% train, 20% test)
train_split = int(0.8 * len(X))  # 80% of data used for training set, 20% for testing
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]

len(X_train), len(y_train), len(X_test), len(y_test)

## 1.4 Visualize the data

A small helper function to plot training data, test data, and (optionally) predictions on the same axes — we'll reuse this throughout the notebook.

| `matplotlib.pyplot` function | What it does |
|---|---|
| `plt.figure(figsize=(w, h))` | starts a new figure of size `w` × `h` inches |
| `plt.scatter(x, y, c, s, label)` | scatter-plots points; `c` = color, `s` = marker size, `label` = legend entry |
| `plt.legend(prop={"size": N})` | draws the legend using the `label=` values above, with font size `N` |


In [ ]:
def plot_predictions(train_data=X_train,
                      train_labels=y_train,
                      test_data=X_test,
                      test_labels=y_test,
                      predictions=None):
  """Plots training data, test data, and (optionally) model predictions."""
  plt.figure(figsize=(10, 7))

  # Plot training data in blue
  plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")

  # Plot test data in green
  plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")

  if predictions is not None:
    # Plot the predictions in red (if they exist)
    plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")

  # Show the legend
  plt.legend(prop={"size": 14});

In [ ]:
plot_predictions();

## 1.5 Build the model (V1) — `nn.Module` + `nn.Parameter`

Almost everything in PyTorch that learns is a subclass of **`nn.Module`**. To build one you need two things:

1. `__init__` — define the *learnable* pieces with `nn.Parameter` (these are the tensors that `requires_grad=True` and get updated during training)
2. `forward()` — define what the model actually *computes* when given input `x`

| Class / method | What it does |
|---|---|
| `nn.Module` | base class for all PyTorch models/layers; handles parameter tracking, `.to(device)`, `.state_dict()`, etc. automatically |
| `nn.Parameter(tensor)` | wraps a tensor so `nn.Module` registers it as a learnable parameter (shows up in `.parameters()`) |
| `super().__init__()` | required first line — sets up the internal `nn.Module` bookkeeping |
| `forward(self, x)` | defines the forward computation; called automatically when you do `model(x)` |

Our model literally implements `y = weight * x + bias`, except `weight` and `bias` start as random numbers instead of `0.7` and `0.3`.


In [ ]:
# Linear regression model, built from raw parameters
class LinearRegressionModelV1(nn.Module):
  def __init__(self):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(1,
                                            requires_grad=True,  # can this parameter be updated via gradient descent?
                                            dtype=torch.float))
    self.bias = nn.Parameter(torch.randn(1,
                                          requires_grad=True,
                                          dtype=torch.float))

  # Forward method defines the computation performed at every call
  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.weight * x + self.bias

# Through gradient descent + backpropagation, the model learns to move
# self.weight and self.bias toward the true weight (0.7) and bias (0.3)

## 1.6 Instantiate the model & inspect its parameters

| Function | What it does |
|---|---|
| `torch.manual_seed(seed)` | fixes the random number generator so `nn.Parameter(torch.randn(...))` produces the *same* "random" values every run — makes results reproducible |
| `model.parameters()` | returns an iterator over all learnable `nn.Parameter` tensors in the model |
| `model.state_dict()` | returns an `OrderedDict` mapping parameter names → their current tensor values (this is exactly what gets saved to disk later) |


In [ ]:
# Create an instance of the model
torch.manual_seed(42)
model_0 = LinearRegressionModelV1()

# Check the parameters
print(list(model_0.parameters()))
print(model_0.state_dict())

## 1.7 Make predictions with the untrained model

Right now `weight` and `bias` are random, so predictions should be way off — that's expected, we haven't trained yet.

| Context manager | What it does |
|---|---|
| `torch.inference_mode()` | disables gradient tracking and some autograd bookkeeping — always wrap prediction/evaluation code in this, it's faster than `torch.no_grad()` and is the modern recommended way to run inference |


In [ ]:
# Make predictions with the model, using torch.inference_mode()
with torch.inference_mode():
  y_preds = model_0(X_test)

y_preds

In [ ]:
plot_predictions(predictions=y_preds)

From the plot above you can see the red prediction dots are nowhere near the green test dots — the model hasn't learned anything yet. That's what training fixes next.


## 1.8 Loss function & optimizer

Two ingredients are needed to actually train a model:

- A **loss function** — measures *how wrong* the predictions are (lower = better)
- An **optimizer** — uses the loss (via gradients) to nudge the model's parameters in a direction that reduces the loss

| Class | What it does | Common args |
|---|---|---|
| `nn.L1Loss()` | Mean Absolute Error: `mean(\|prediction - target\|)` | — |
| `torch.optim.SGD(params, lr, momentum)` | Stochastic Gradient Descent optimizer | `params` = what to update (`model.parameters()`), `lr` = learning rate (step size), `momentum` = accelerates SGD in the right direction |


In [ ]:
# Create the loss function
loss_fn = nn.L1Loss()

# Create the optimizer
optimizer = torch.optim.SGD(params=model_0.parameters(),
                             lr=0.01,
                             momentum=0.9)

## 1.9 Build a training loop

The standard PyTorch training loop has five steps, done every epoch:

| Step | Code | What it does |
|---|---|---|
| 1. Forward pass | `y_pred = model(X_train)` | run the model on the training data |
| 2. Calculate loss | `loss = loss_fn(y_pred, y_train)` | compare predictions to the truth |
| 3. Zero the gradients | `optimizer.zero_grad()` | gradients accumulate by default — reset them each step |
| 4. Backpropagation | `loss.backward()` | compute gradients of the loss w.r.t. every parameter |
| 5. Optimizer step | `optimizer.step()` | update parameters using those gradients |

Each epoch we also run a quick evaluation pass on the *test* set (inside `torch.inference_mode()`, no gradients needed) so we can track how well the model generalizes.

| Method | What it does |
|---|---|
| `model.train()` | puts the model in **training** mode (relevant for layers like Dropout/BatchNorm — no effect here, but it's good habit) |
| `model.eval()` | puts the model in **evaluation** mode |


In [ ]:
torch.manual_seed(42)

epochs = 200
epoch_count = []
loss_values = []
test_loss_values = []

for epoch in range(epochs):
    ### Training
    model_0.train()  # train mode is on by default after construction

    # 1. Forward pass
    y_pred = model_0(X_train)

    # 2. Calculate loss
    loss = loss_fn(y_pred, y_train)

    # 3. Zero grad optimizer
    optimizer.zero_grad()

    # 4. Loss backward (perform backpropagation)
    loss.backward()

    # 5. Step the optimizer
    optimizer.step()

    ### Testing
    model_0.eval()  # put the model in evaluation mode for testing (inference)
    with torch.inference_mode():  # turns off gradient tracking
        test_pred = model_0(X_test)
        test_loss = loss_fn(test_pred, y_test)

    # Print out what's happening every 100 epochs
    if epoch % 100 == 0:
        epoch_count.append(epoch)
        loss_values.append(loss.item())
        test_loss_values.append(test_loss.item())
        print(f"Epoch: {epoch} | Train loss: {loss} | Test loss: {test_loss}")

## 1.10 Compare the learned parameters to the originals

| Function | What it does |
|---|---|
| `pprint.pprint(obj)` | "pretty print" — formats nested dicts/lists more readably than a plain `print()` |


In [ ]:
# Find our model's learned parameters
from pprint import pprint  # pprint = pretty print, see: https://docs.python.org/3/library/pprint.html
print("The model learned the following values for weights and bias:")
pprint(model_0.state_dict())
print("\nAnd the original values for weights and bias are:")
print(f"weights: {weight}, bias: {bias}")

After ~200 epochs, `model_0`'s learned `weight`/`bias` should be very close to the true `0.7` / `0.3` — not exact (L1Loss with a fairly high learning rate won't converge perfectly), but close enough to see the model has genuinely learned the pattern.


## 1.11 Final predictions & visualization

Now that `model_0` is trained, let's generate one more set of predictions in eval mode and plot them.


In [ ]:
# Turn model into evaluation mode
model_0.eval()

# Make predictions on the test data
with torch.inference_mode():
    y_preds = model_0(X_test)
y_preds

In [ ]:
# Put data on the CPU and plot it
# (.cpu() is a no-op here since everything is already on CPU, but it's a good
#  habit for GPU workflows — matplotlib can only plot CPU/numpy data)
plot_predictions(predictions=y_preds.cpu())

The red prediction dots should now sit right on top of the green test dots — the model has learned the line.


## 1.12 Saving and loading the model

PyTorch gives you three core functions for model persistence:

| Function | What it does |
|---|---|
| `torch.save(obj, f)` | serializes a Python object (commonly a `state_dict`) to disk using Python's `pickle` format |
| `torch.load(f)` | deserializes a saved object back into memory |
| `model.load_state_dict(state_dict)` | loads a `state_dict` (parameter names → tensors) into an existing model's parameters |

The recommended pattern is to save the **`state_dict()`** (just the learned numbers) rather than the whole model object — it's smaller, more portable, and doesn't tie you to the exact class definition/file path used when saving.

> ℹ️ Since PyTorch 2.6, `torch.load()` defaults to `weights_only=True` for security (it restricts what can be unpickled). That's exactly what we want here since we're only ever saving/loading a plain `state_dict` of tensors, so no extra arguments are needed.

We also use `pathlib.Path` to build the save path in an OS-independent way:

| `pathlib` | What it does |
|---|---|
| `Path("models")` | creates a `Path` object representing the folder `models` |
| `Path.mkdir(parents=True, exist_ok=True)` | creates the directory (and any missing parent folders); doesn't error if it already exists |


In [ ]:
from pathlib import Path

# 1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_0.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_0.state_dict(),  # only saving the state_dict() saves just the learned parameters
           f=MODEL_SAVE_PATH)

In [ ]:
# Instantiate a fresh instance of LinearRegressionModelV1
loaded_model_0 = LinearRegressionModelV1()

# Load the saved state_dict into the new instance
loaded_model_0.load_state_dict(torch.load(MODEL_SAVE_PATH))

print(f"Loaded model:\n{loaded_model_0}")
print(f"Model on device:\n{next(loaded_model_0.parameters()).device}")

# Evaluate the loaded model
loaded_model_0.eval()
with torch.inference_mode():
    loaded_model_0_preds = loaded_model_0(X_test)

# Should be all True: the reloaded model produces identical predictions
y_preds == loaded_model_0_preds

`next(model.parameters()).device` is a common idiom: `.parameters()` returns a *generator*, and `next()` grabs the first item from it — an easy way to check what device a model's weights currently live on without looping over all of them.


---
# Part 2 — Putting it all together (the idiomatic way)

Same workflow, same idea — but this time written the way you'd actually write PyTorch code day-to-day:
- **Device-agnostic**: the code automatically uses a GPU if one's available, and CPU otherwise
- **`nn.Linear`** instead of hand-rolled `nn.Parameter`s — PyTorch's built-in linear layer does the same `weight * x + bias` computation (plus proper weight initialization) for you


In [ ]:
### PUTTING IT ALL TOGETHER
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn

# Check the version
torch.__version__

## 2.1 Device-agnostic code

| Function | What it does |
|---|---|
| `torch.cuda.is_available()` | returns `True` if a CUDA-capable GPU is visible to PyTorch |

Writing `device = "cuda" if torch.cuda.is_available() else "cpu"` once at the top means every `.to(device)` call below "just works" whether you're on a laptop CPU or a cloud GPU — no code changes needed.


In [ ]:
# Setting up device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 2.2 Create data

Same idea as Part 1 — a fresh `weight`/`bias` pair (different values this time, `0.4` / `0.3`, just to prove this is an independent run) and a synthetic straight line.


In [ ]:
# Create weight and bias for Y = weight * X + bias
weight = 0.4
bias = 0.3

start = 0
end = 1
step = 0.02

# Create X and y (features and labels)
X = torch.arange(start=start, end=end, step=step).unsqueeze(dim=1)
y = weight * X + bias

X[:10], y[:10]

## 2.3 Split into train/test sets


In [ ]:
# Create train/test split (80% train, 20% test)
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]
len(X_train), len(y_train), len(X_test), len(y_test)

In [ ]:
# Plot the data (uses the plot_predictions() helper defined in Part 1)
plot_predictions(X_train, y_train, X_test, y_test)

## 2.4 Build the model (V2) — `nn.Linear`

Instead of manually creating `nn.Parameter` tensors, we use **`nn.Linear`**, PyTorch's built-in fully-connected layer. `nn.Linear(in_features=1, out_features=1)` internally holds a `weight` tensor and a `bias` tensor — mathematically identical to what we built by hand in Part 1 — but with sensible default initialization and a highly-optimized `forward()`.

| Class | Signature | What it does |
|---|---|---|
| `nn.Linear(in_features, out_features)` | `in_features` = size of each input sample, `out_features` = size of each output sample | Applies `y = xW^T + b`; for `in_features=out_features=1` this reduces to our familiar `y = weight * x + bias` |


In [ ]:
class LinearRegressionModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        # Define a single linear layer with 1 input feature and 1 output feature
        self.linear_layer = nn.Linear(in_features=1, out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)

# Set the manual seed
torch.manual_seed(42)
model_1 = LinearRegressionModelV2()
model_1, model_1.state_dict()

In [ ]:
# Check what device the model's parameters are currently on
next(model_1.parameters()).device

## 2.5 Move the model *and the data* to the target device


In [ ]:
# Set the model to use the target device
model_1.to(device)

# Put the data on the target device too
X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

next(model_1.parameters()).device

## 2.6 Loss function & optimizer


In [ ]:
# Create the loss function
loss_fn = nn.L1Loss()

# Create the optimizer
optimizer = torch.optim.SGD(params=model_1.parameters(),
                             lr=0.01)

## 2.7 Training loop

Same five-step loop as Part 1.


In [ ]:
torch.manual_seed(42)

epochs = 1000
epoch_count = []
loss_values = []
test_loss_values = []

for epoch in range(epochs):
    model_1.train()

    # 1. Forward pass
    y_pred = model_1(X_train)

    # 2. Calculate the loss
    loss = loss_fn(y_pred, y_train)

    # 3. Zero grad optimizer
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    ### Testing
    model_1.eval()
    with torch.inference_mode():
        test_pred = model_1(X_test)
        test_loss = loss_fn(test_pred, y_test)

    # Print what's happening every 100 epochs
    if epoch % 100 == 0:
        epoch_count.append(epoch)
        loss_values.append(loss.item())
        test_loss_values.append(test_loss.item())
        print(f"Epoch: {epoch} | Train loss: {loss} | Test loss: {test_loss}")

## 2.8 Inspect the learned parameters


In [ ]:
model_1.state_dict()

In [ ]:
print(f"Original values -> weight: {weight}, bias: {bias}")
weight, bias

## 2.9 Save and reload `model_1`

In [ ]:
from pathlib import Path

# 1. Create models directory (re-use if it already exists)
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_1.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model state_dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_1.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
# Instantiate a fresh instance of LinearRegressionModelV2
loaded_model_1 = LinearRegressionModelV2()

# Load the saved state_dict
loaded_model_1.load_state_dict(torch.load(MODEL_SAVE_PATH))

# Put the loaded model on the target device
loaded_model_1.to(device)

print(f"Loaded model:\n{loaded_model_1}")
print(f"Model on device:\n{next(loaded_model_1.parameters()).device}")

# Evaluate loaded model
loaded_model_1.eval()
with torch.inference_mode():
    loaded_model_1_preds = loaded_model_1(X_test)

with torch.inference_mode():
    original_preds = model_1(X_test)

# Should be all True
original_preds == loaded_model_1_preds

---
# 📋 Cheat Sheet — every function/class used in this notebook

A single reference table for quick revision. Grouped by workflow stage.

### Data creation & tensors
| Function | Signature | What it does |
|---|---|---|
| `torch.arange` | `torch.arange(start, end, step)` | 1‑D tensor of evenly spaced values on `[start, end)` |
| `Tensor.unsqueeze` | `tensor.unsqueeze(dim)` | inserts a size‑1 dimension at `dim` |
| `Tensor.squeeze` | `tensor.squeeze(dim=None)` | removes size‑1 dimensions (⚠️ can cause broadcasting bugs if used carelessly before a loss calc — see fix #6 above) |
| `torch.manual_seed` | `torch.manual_seed(seed)` | seeds the RNG for reproducible "random" tensors |

### Building models
| Function | Signature | What it does |
|---|---|---|
| `nn.Module` | base class | parent class for all models/layers; auto-tracks parameters |
| `nn.Parameter` | `nn.Parameter(tensor)` | marks a tensor as a learnable, trainable parameter |
| `nn.Linear` | `nn.Linear(in_features, out_features)` | fully-connected layer computing `xW^T + b` |
| `model.forward(x)` | called via `model(x)` | defines the forward computation |
| `model.parameters()` | — | iterator over all learnable parameters |
| `model.state_dict()` | — | `OrderedDict` of parameter name → tensor value |

### Training
| Function | Signature | What it does |
|---|---|---|
| `nn.L1Loss` | `nn.L1Loss()` | Mean Absolute Error loss: `mean(\|pred - target\|)` |
| `torch.optim.SGD` | `SGD(params, lr, momentum=0)` | Stochastic Gradient Descent optimizer |
| `optimizer.zero_grad()` | — | resets accumulated gradients to zero before backprop |
| `loss.backward()` | — | computes gradients of `loss` w.r.t. every parameter (autograd) |
| `optimizer.step()` | — | updates parameters using the gradients just computed |
| `model.train()` | — | sets the model to training mode |
| `model.eval()` | — | sets the model to evaluation mode |
| `torch.inference_mode()` | context manager | disables gradient tracking for faster inference |

### Device-agnostic code
| Function | Signature | What it does |
|---|---|---|
| `torch.cuda.is_available()` | — | `True` if a CUDA GPU is available |
| `model.to(device)` / `tensor.to(device)` | `.to("cpu"/"cuda")` | moves a model or tensor to the given device |

### Saving & loading
| Function | Signature | What it does |
|---|---|---|
| `torch.save` | `torch.save(obj, f)` | pickles/serializes an object to disk |
| `torch.load` | `torch.load(f)` | loads a previously-saved object (defaults to `weights_only=True` since PyTorch 2.6) |
| `model.load_state_dict` | `model.load_state_dict(state_dict)` | loads saved parameter values into a model instance |
| `pathlib.Path` | `Path("folder")` | OS-independent filesystem path object |
| `Path.mkdir` | `path.mkdir(parents=True, exist_ok=True)` | creates a directory (and parents) if it doesn't already exist |

### Visualization
| Function | Signature | What it does |
|---|---|---|
| `plt.figure` | `plt.figure(figsize=(w, h))` | starts a new figure |
| `plt.scatter` | `plt.scatter(x, y, c, s, label)` | scatter plot |
| `plt.legend` | `plt.legend(prop={"size": N})` | draws a legend from `label=` entries |

### Misc Python
| Function | What it does |
|---|---|
| `next(iterator)` | pulls the first/next item from an iterator (used on `model.parameters()`) |
| `pprint.pprint(obj)` | pretty-prints nested data structures |

---
### 📚 References
- [01. PyTorch Workflow Fundamentals — learnpytorch.io](https://www.learnpytorch.io/01_pytorch_workflow/)
- [PyTorch documentation (v2.13)](https://docs.pytorch.org/docs/2.13/index.html)
